# 3. Batch Inference and Final Outputs

This notebook runs a SageMaker Batch Transform job and treats S3 as the durable final-output store. It also shows where evaluation metrics and run metadata can be written for later analysis.

In [ ]:
import json
import time
import boto3

REGION = "us-east-1"
BUCKET = "replace-with-your-dev-bucket"
MODEL_NAME = "replace-with-created-sagemaker-model"
sm = boto3.client("sagemaker", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)

## Submit batch inference

The input prefix contains records in the format expected by the model. SageMaker writes prediction files below the output prefix.

In [ ]:
job_name = f"taxi-batch-{int(time.time())}"
output_prefix = f"s3://{BUCKET}/inference/dev/{job_name}/"
sm.create_transform_job(
    TransformJobName=job_name,
    ModelName=MODEL_NAME,
    TransformInput={"DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": f"s3://{BUCKET}/inference/input/"}}},
    TransformOutput={"S3OutputPath": output_prefix, "AssembleWith": "Line"},
    TransformResources={"InstanceType": "ml.m5.large", "InstanceCount": 1},
)
print(output_prefix)

In [ ]:
sm.get_waiter("transform_job_completed_or_stopped").wait(TransformJobName=job_name)
job = sm.describe_transform_job(TransformJobName=job_name)
print(job["TransformJobStatus"])
print(job.get("TransformOutput", {}).get("S3OutputPath"))

## Store run metadata

Keep predictions, metrics, and a small manifest together. A catalog or warehouse can be added later if downstream consumers need SQL, dashboards, or retention policies.

In [ ]:
manifest = {"job_name": job_name, "model_name": MODEL_NAME, "output_prefix": output_prefix, "status": job["TransformJobStatus"]}
s3.put_object(
    Bucket=BUCKET,
    Key=f"inference/dev/{job_name}/manifest.json",
    Body=json.dumps(manifest).encode("utf-8"),
    ContentType="application/json",
)
print(manifest)